# 04 — Stage-2 交互式训练(xc 口径)

在 notebook 里跑 hyperopt + 终训 + 评估,便于探索;可复现的生产训练用 `scripts/run_training.py`(runbook 第 4 步)。内核须带 xgboost(本机 conda `env_ml`)。

xc 要点:时间切分 70/15/15 + `cv_strategy: time_forward`(前向扩窗时间折,与最终时间切分口径一致)+ `tuner_objective: precision_at_k`(调参直接最大化 CV P@10%,不是全曲线 PR-AUC);`xc_qual_finish_1v1` / `xc_e2e_credit_1v1` 另有 credit_1v1 档位价值加权 —— 权重只进训练损失,调参 P@K、早停与上报指标都按人头不加权。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from wdm.config import load_config
from wdm.model.dataset import build_dataset
from wdm.model.tuner import run_hyperopt
from wdm.model.trainer import train_final
from wdm.model.evaluator import evaluate_all

PRODUCT = 'xc_resp_finish'
FEATURES_VERSION = None   # None = config 的 active_version;生产链路应是 v3_manual_noleak
MAX_EVALS = 10            # 探索用;生产走 run_training.py(配置默认 30)

cfg = load_config(PRODUCT)
tr = cfg['training']
print('tuner_objective={0} | cv_strategy={1} | split={2} {3} | top_k_pct={4}'.format(
    tr.get('tuner_objective'), tr.get('cv_strategy'),
    tr['split']['strategy'], tr['split']['ratios'], tr.get('top_k_pct')))
print('sample_weight:', tr.get('sample_weight'))
data = build_dataset(cfg, version=FEATURES_VERSION)
print('train/valid/oot shapes:', data.X_train.shape, data.X_valid.shape, data.X_oot.shape)

In [ ]:
# 切分体检:正例率跨切分漂移大 → 时间漂移预警;dt 区间须前后相接不重叠;
# 1v1 产品的 w_train 应只对正样本档位 > 1(负样本/档位 1 均为 1.0)
rows = []
for split, y, dt in [('train', data.y_train, data.dt_train),
                     ('valid', data.y_valid, data.dt_valid),
                     ('oot', data.y_oot, data.dt_oot)]:
    row = {'split': split, 'n': len(y), 'pos_rate': round(float(np.mean(y)), 4)}
    if dt is not None:
        row['dt_min'], row['dt_max'] = int(np.nanmin(dt)), int(np.nanmax(dt))
    rows.append(row)
display(pd.DataFrame(rows))
if data.w_train is not None:
    wdist = (pd.DataFrame({'label': data.y_train, 'w': np.round(data.w_train, 4)})
             .groupby(['label', 'w']).size().rename('n').reset_index())
    print('w_train 分布(按 label x 权重):')
    display(wdist)
else:
    print('无 sample_weight(仅 1v1 两个产品配置了档位价值加权)')

In [ ]:
# 与 run_training.py 一致:dt_train 供 cv_strategy='time_forward' 用,w_train 供 sample_weight 用
# (bank_marketing 两者皆 None,不影响;xc 系产品必须传)
best_params, best_loss, trials = run_hyperopt(
    data.X_train, data.y_train, cfg, max_evals=MAX_EVALS,
    dt_train=data.dt_train, w_train=data.w_train)
OBJ = trials.trials[0]['result'].get('objective', 'aucpr')
print('best params:', best_params)
print('best CV {0}:'.format(OBJ), -best_loss)

In [ ]:
# hyperopt 轨迹
losses = [t['result']['loss'] for t in trials.trials]
plt.plot(range(1, len(losses) + 1), [-l for l in losses], marker='o')
plt.xlabel('trial')
plt.ylabel('CV {0}'.format(OBJ))
plt.title('Hyperopt trajectory')
plt.grid(alpha=0.3)

In [ ]:
# 早停看无权重 valid 指标;权重只进训练损失(与 run_training.py 一致)
booster, evals = train_final(best_params, data.X_train, data.y_train,
                             data.X_valid, data.y_valid, cfg, w_tr=data.w_train)
metrics_df, binned, scores, imp = evaluate_all(booster, data, cfg)
metrics_df

In [ ]:
# 训练后快速诊断:Train -> Valid -> OOT 退化检查(runbook 第 4 步:KS/PR-AUC 腰斩
# 即怀疑过拟合/时间漂移)+ 分位累计 lift + 头部重要性
m = metrics_df.set_index('split')
for col in ['pr_auc', 'ks', 'lift_at_k']:
    tr_v, oot_v = float(m.loc['train', col]), float(m.loc['oot', col])
    ratio = oot_v / tr_v if tr_v else float('nan')
    note = '   <-- 退化过半!' if ratio < 0.5 else ''
    print('{0}: train={1:.4f} valid={2:.4f} oot={3:.4f} | oot/train={4:.2f}{5}'.format(
        col, tr_v, float(m.loc['valid', col]), oot_v, ratio, note))

fig, ax = plt.subplots(figsize=(7, 4))
for split in ['valid', 'oot']:
    b = binned[split]
    ax.plot(b['bin'], b['cum_lift'], marker='o', label=split)
ax.axhline(1.0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('decile (1 = highest score)'); ax.set_ylabel('cumulative lift')
ax.set_title('cumulative lift by depth'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

print('importance top 15:')
display(imp.head(15))